In [2]:
!pip install requests
!pip install pymysql


import requests

url = " https://api-web.nhle.com/v1/standings/now"
response = requests.get(url)
data= response.json()


import pymysql
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    database='nhl'
)

cursor = conn.cursor()



standings_records=[]
for team in data['standings']:
        
        team_abbrev = team['teamAbbrev']
        if isinstance(team_abbrev, dict):
                team_abbrev = team_abbrev['default']
        season_Id = team['seasonId']
        games_played = team['gamesPlayed']
        wins=team['wins']
        losses=team['losses']
        ot_losses=team['otLosses']
        points=team['points']
        goals_for=team['goalFor']
        goals_against=team['goalAgainst']
        home_wins=team['homeWins']
        away_wins=team['roadWins']
        streak_type=team['streakCode']
        streak_count=team['streakCount']
        standings_records.append((team_abbrev, season_Id, games_played, wins, losses, ot_losses, points, goals_for, goals_against, home_wins, away_wins, streak_type, streak_count))
standings_records   



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[('COL', 20252026, 82, 55, 16, 11, 121, 302, 203, 26, 29, 'W', 3),
 ('CAR', 20252026, 82, 53, 22, 7, 113, 296, 240, 29, 24, 'W', 1),
 ('DAL', 20252026, 82, 50, 20, 12, 112, 279, 226, 26, 24, 'W', 5),
 ('BUF', 20252026, 82, 50, 23, 9, 109, 288, 241, 26, 24, 'OT', 1),
 ('TBL', 20252026, 82, 50, 26, 6, 106, 290, 231, 26, 24, 'L', 1),
 ('MTL', 20252026, 82, 48, 24, 10, 106, 283, 256, 24, 24, 'L', 1),
 ('MIN', 20252026, 82, 46, 24, 12, 104, 272, 240, 23, 23, 'W', 1),
 ('BOS', 20252026, 82, 45, 27, 10, 100, 272, 250, 29, 16, 'W', 2),
 ('OTT', 20252026, 82, 44, 27, 11, 99, 278, 246, 23, 21, 'W', 1),
 ('PIT', 20252026, 82, 41, 25, 16, 98, 293, 268, 20, 21, 'L', 3),
 ('PHI', 20252026, 82, 43, 27, 12, 98, 250, 243, 20, 23, 'W', 3),
 ('WSH', 20252026, 82, 43, 30, 9, 95, 263, 244, 25, 18, 'W', 4),
 ('VGK', 20252026, 82, 39, 26, 17, 95, 265, 250, 20, 19, 'W', 3),
 ('EDM', 20252026, 82, 41, 30, 11, 93, 282, 269, 22, 19, 'W', 1),
 ('UTA', 20252026, 82, 43, 33, 6, 92, 268, 240, 22, 21, 'L', 1),
 ('DET

In [3]:
create_table_query = """
CREATE TABLE IF NOT EXISTS standings (
    standing_id INT AUTO_INCREMENT PRIMARY KEY,
    team_id INT NOT NULL,
    season VARCHAR(20),
    games_played INT,
    wins INT,
    losses INT,
    ot_losses INT,
    points INT,
    goals_for INT,
    goals_against INT,
    home_wins INT,
    away_wins INT,
    streak_type VARCHAR(20),
    streak_count INT,
    CONSTRAINT fk_team FOREIGN KEY (team_id) REFERENCES teams(team_id)
);
"""
cursor.execute(create_table_query)

0

In [4]:
insert_query = """
INSERT INTO standings (
    team_id,
    season,
    games_played,
    wins,
    losses,
    ot_losses,
    points,
    goals_for,
    goals_against,
    home_wins,
    away_wins,
    streak_type,
    streak_count
)
SELECT
    team_id,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s
FROM teams
WHERE team_abbrev = %s
"""

for record in standings_records:
    (
        team_abbrev,
        season_Id,
        games_played,
        wins,
        losses,
        ot_losses,
        points,
        goals_for,
        goals_against,
        home_wins,
        away_wins,
        streak_type,
        streak_count,
    ) = record

    cursor.execute(
        insert_query,
        (
            season_Id,
            games_played,
            wins,
            losses,
            ot_losses,
            points,
            goals_for,
            goals_against,
            home_wins,
            away_wins,
            streak_type,
            streak_count,
            team_abbrev,
        ),
    )

conn.commit()
